# 🔗 Day 04a: Chain of Responsibility & Mediator Patterns

---

## 🎯 What You'll Master Today
- **Chain of Responsibility**: pass request along a chain of handlers until one handles it
- **Mediator**: centralize communication between objects to reduce coupling
- Real-world: middleware pipelines, logging levels, chat rooms, air traffic control

---

## 🎭 The Analogies

**Chain of Responsibility** = Customer support escalation  
You call support → L1 agent tries → can't solve → escalates to L2 → can't solve → L3 manager handles it. Each handler either handles or passes along.

**Mediator** = Air traffic control tower  
Planes don't talk to each other directly. The tower (mediator) coordinates everyone. Planes only know the tower, not each other.

---

In [ ]:
# ============================================
# 1. CHAIN OF RESPONSIBILITY — Middleware
# ============================================
from abc import ABC, abstractmethod

class Handler(ABC):
    def __init__(self):
        self._next: 'Handler' = None
    
    def set_next(self, handler: 'Handler') -> 'Handler':
        self._next = handler
        return handler  # enables chaining
    
    def handle(self, request: dict) -> str:
        if self._next:
            return self._next.handle(request)
        return "❌ No handler could process the request"


class AuthHandler(Handler):
    def handle(self, request: dict) -> str:
        if not request.get("token"):
            return "🚫 AUTH: No token provided"
        print(f"  ✅ Auth: Token verified")
        return super().handle(request)

class RateLimitHandler(Handler):
    def __init__(self, max_requests: int = 100):
        super().__init__()
        self._count = 0
        self._max = max_requests
    
    def handle(self, request: dict) -> str:
        self._count += 1
        if self._count > self._max:
            return "🚫 RATE LIMIT: Too many requests"
        print(f"  ✅ Rate limit: {self._count}/{self._max}")
        return super().handle(request)

class ValidationHandler(Handler):
    def handle(self, request: dict) -> str:
        if not request.get("body"):
            return "🚫 VALIDATION: Empty request body"
        print(f"  ✅ Validation: Body present")
        return super().handle(request)

class BusinessHandler(Handler):
    def handle(self, request: dict) -> str:
        return f"✅ PROCESSED: {request['body']}"


# Build chain: Auth → RateLimit → Validation → Business
auth = AuthHandler()
rate = RateLimitHandler(max_requests=5)
validate = ValidationHandler()
business = BusinessHandler()

auth.set_next(rate).set_next(validate).set_next(business)

# Test cases
print("1. Valid request:")
print(auth.handle({"token": "abc123", "body": "Create user"}))

print("\n2. No token:")
print(auth.handle({"body": "Create user"}))

print("\n3. No body:")
print(auth.handle({"token": "abc123"}))

In [ ]:
# ============================================
# 2. Pythonic Chain — Using List of Handlers
# ============================================
from typing import Callable, Optional

class Pipeline:
    """Pythonic middleware pipeline."""
    
    def __init__(self):
        self._handlers: list[Callable] = []
    
    def add(self, handler: Callable) -> 'Pipeline':
        self._handlers.append(handler)
        return self
    
    def execute(self, data: dict) -> dict:
        for handler in self._handlers:
            result = handler(data)
            if result is not None:  # handler responded = stop
                return result
        return {"error": "No handler processed the request"}


# Handlers as functions
def check_auth(req):
    if not req.get("user"):
        return {"status": 401, "error": "Unauthorized"}

def check_permission(req):
    if req.get("user") != "admin":
        return {"status": 403, "error": "Forbidden"}

def process_request(req):
    return {"status": 200, "data": f"Welcome, {req['user']}!"}


pipe = Pipeline()
pipe.add(check_auth).add(check_permission).add(process_request)

print(pipe.execute({"user": "admin"}))   # 200
print(pipe.execute({"user": "guest"}))   # 403
print(pipe.execute({}))                   # 401

In [ ]:
# ============================================
# 3. MEDIATOR — Chat Room
# ============================================
from abc import ABC, abstractmethod

class ChatMediator(ABC):
    @abstractmethod
    def send(self, message: str, sender: 'User') -> None: pass
    @abstractmethod
    def add_user(self, user: 'User') -> None: pass

class User:
    def __init__(self, name: str, mediator: ChatMediator):
        self.name = name
        self._mediator = mediator
        mediator.add_user(self)
    
    def send(self, message: str):
        print(f"  {self.name} sends: {message}")
        self._mediator.send(message, self)
    
    def receive(self, message: str, sender: str):
        print(f"  {self.name} received from {sender}: {message}")


class ChatRoom(ChatMediator):
    """Concrete mediator — routes messages between users."""
    
    def __init__(self, name: str):
        self.name = name
        self._users: list[User] = []
    
    def add_user(self, user: User):
        self._users.append(user)
        print(f"  📢 {user.name} joined #{self.name}")
    
    def send(self, message: str, sender: User):
        for user in self._users:
            if user != sender:  # don't send to yourself
                user.receive(message, sender.name)


# Users don't know about each other — only the mediator
room = ChatRoom("general")
alice = User("Alice", room)
bob = User("Bob", room)
charlie = User("Charlie", room)

print()
alice.send("Hello everyone!")
print()
bob.send("Hey Alice!")

```
╔═══════════════════════════════════════════════════════════╗
║             PATTERN COMPARISON MAP                        ║
╠═══════════════════╦═══════════════════════════════════════╣
║  Aspect            ║  Chain of Resp. vs Mediator          ║
╠═══════════════════╬═══════════════════════════════════════╣
║  Communication     ║  Chain: sequential pass-along         ║
║                    ║  Mediator: centralized hub            ║
║  Who handles?      ║  Chain: first capable handler         ║
║                    ║  Mediator: routes to all/specific     ║
║  Coupling          ║  Chain: each knows next handler       ║
║                    ║  Mediator: components know mediator   ║
║  Use case          ║  Chain: middleware, validation        ║
║                    ║  Mediator: chat, event systems        ║
╚═══════════════════╩═══════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | Chain of Responsibility in web frameworks? | Django/FastAPI middleware. Request passes through auth → logging → routing → handler |
| 2 | Can multiple handlers in a chain process the same request? | Yes — variant where each handler processes AND passes. Like logging middleware |
| 3 | Mediator vs Observer? | Mediator: 1 central coordinator. Observer: multiple direct subscriptions. Mediator is more controlled |
| 4 | When does Mediator become a God Object? | When it does too much business logic instead of just routing. Keep it thin |
| 5 | Real-world Chain examples? | Exception handling (try/except chain), event bubbling in DOM, servlet filters |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Chain = pass request until someone handles it         │
## │  • Mediator = central hub for component communication    │
## │  • Chain: linked list of handlers (middleware pattern)   │
## │  • Mediator: star topology (components ↔ mediator)       │
## │  • Both reduce direct coupling between objects            │
## │  • Pythonic chain: just use a list of callables           │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## 🎉 Congratulations! You've completed all OOP & Design Patterns!
## Time to practice: pick a pattern and build something real.